# LoRA: Low-Rank Adaptation - 실습 코드 2: LoRA 모듈을 직접 구현하기 (PyTorch)

- Tutorial ID: `expand-lora`
- Tutorial: LoRA: Low-Rank Adaptation
- Section ID: `expand-lora-code-2`
- Section: 실습 코드 2: LoRA 모듈을 직접 구현하기 (PyTorch)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: LoRA 모듈을 직접 구현하기 (PyTorch)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 큰 가중치 W를 직접 바꾸지 않고 저랭크 A/B 업데이트가 어떻게 더해지는지 확인
#   2) 위치 정보가 벡터 회전/편향으로 attention score에 들어가는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    """LoRA: Low-Rank Adaptation을 직접 구현
    
    논문: W = W_0 + ΔW = W_0 + B·A
    - W_0: frozen pre-trained weight
    - B·A: trainable low-rank decomposition
    """
        def __init__(self, original_linear: nn.Linear, rank: int = 8, 
                 alpha: float = 16.0, dropout: float = 0.0):
        super().__init__()
        self.original = original_linear
        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank
        
        d_out, d_in = original_linear.weight.shape
        
        # LoRA 가중치
        self.lora_A = nn.Parameter(torch.randn(d_in, rank) * math.sqrt(2.0 / d_in))
        self.lora_B = nn.Parameter(torch.zeros(rank, d_out))
        
        # Dropout
                self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        
        # 원래 가중치는 동결
        self.original.weight.requires_grad = False
        if self.original.bias is not None:
            self.original.bias.requires_grad = False
    
        def forward(self, x):
        # 원래 선형 변환 (frozen)
                original_out = self.original(x)
        
        # LoRA 경로: x @ A @ B * scale
                lora_out = self.dropout(x) @ self.lora_A @ self.lora_B.T * self.scale
        
        return original_out + lora_out
    
        def merge_weights(self):
        """학습 후 LoRA 가중치를 원래 가중치에 병합"""
        with torch.no_grad():
            delta_W = (self.lora_B.T @ self.lora_A.T) * self.scale
            self.original.weight.data += delta_W
        return self.original


# ── LoRA 적용 예시 ──
def apply_lora_to_model(model, rank=8, alpha=16.0, target_names=None):
    """모델의 특정 레이어에 LoRA 적용"""
    if target_names is None:
        target_names = ["q_proj", "k_proj", "v_proj", "o_proj"]
    
    lora_params = []
    total_params = 0
    
        for name, module in model.named_modules():
        if any(t in name for t in target_names) and isinstance(module, nn.Linear):
            # LoRA로 교체
            lora_module = LoRALinear(module, rank=rank, alpha=alpha)
            
            # 모듈 교체 (계층 구조 유지)
            parts = name.split('.')
            parent = model
                        for part in parts[:-1]:
                parent = getattr(parent, part)
            setattr(parent, parts[-1], lora_module)
            
            # LoRA 파라미터 수집
            lora_params.extend([lora_module.lora_A, lora_module.lora_B])
            n_lora = lora_module.lora_A.numel() + lora_module.lora_B.numel()
            total_params += n_lora
            print(f"  LoRA 적용: {name} "
                  f"({module.weight.shape} → rank={rank}, params={n_lora:,})")
    
    print(f"\n총 LoRA 파라미터: {total_params:,}")
    return model, lora_params


# ── LoRA 학습 루프 ──
def train_with_lora(model, lora_params, train_loader, epochs=3, lr=1e-4):
    """LoRA 파라미터만 학습"""
    # LoRA 파라미터만 optimizer에 전달
    optimizer = torch.optim.AdamW(lora_params, lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    model.train()
        for epoch in range(epochs):
                total_loss = 0
                for batch in train_loader:
            input_ids, labels = batch
                        logits = model(input_ids)
                        loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1)
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
                avg_loss = total_loss / len(train_loader)
                print(f"Epoch {epoch+1}: loss={avg_loss:.4f}")
        scheduler.step()


# ── 사용 예시 ──
# 작은 테스트 모델
class SmallLLM(nn.Module):
        def __init__(self, vocab_size=32000, d_model=512, n_layers=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList()
                for _ in range(n_layers):
            self.layers.append(nn.ModuleDict({
                "q_proj": nn.Linear(d_model, d_model),
                "k_proj": nn.Linear(d_model, d_model),
                "v_proj": nn.Linear(d_model, d_model),
                "o_proj": nn.Linear(d_model, d_model),
                "ffn": nn.Sequential(
                    nn.Linear(d_model, d_model * 4),
                    nn.GELU(),
                    nn.Linear(d_model * 4, d_model),
                ),
            }))
    
        def forward(self, x):
                x = self.embed(x)
                for layer in self.layers:
                        x = layer["o_proj"](layer["q_proj"](x)) + x  # 단순화
                        x = layer["ffn"](x) + x
        return x

model = SmallLLM()
total_before = sum(p.numel() for p in model.parameters())
print(f"모델 파라미터: {total_before:,}")

model, lora_params = apply_lora_to_model(model, rank=8, alpha=16.0)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"학습 가능 파라미터: {trainable:,} ({trainable/total_before*100:.2f}%)")